# 13. openFDA API EDA (T0-4)

**데이터 출처**: https://api.fda.gov (인증 불필요)  
**파이프라인 역할**: 부작용/라벨 구조 sanity check, 보고서 보조 참조

## EDA 체크리스트
- [ ] 주요 엔드포인트 응답 구조 확인 (`/drug/label`, `/drug/event`)
- [ ] 한국 성분명 50종 → FDA 라벨 검색 히트율
- [ ] 부작용 보고 건수 분포
- [ ] 산출물: `openfda_summary.json`

In [6]:
import json, time
import requests
import pandas as pd
from pathlib import Path

ROOT = Path("../../").resolve()
INTERIM = ROOT / "data" / "interim"

BASE = "https://api.fda.gov"

In [7]:
def openfda_get(endpoint, params, timeout=15):
    resp = requests.get(f"{BASE}{endpoint}", params=params, timeout=timeout)
    if resp.status_code == 404:
        return None
    resp.raise_for_status()
    return resp.json()

# 연결 확인
r = openfda_get("/drug/label.json", {"search": "openfda.generic_name:aspirin", "limit": 1})
print("총 aspirin 라벨:", r["meta"]["results"]["total"])

총 aspirin 라벨: 715


In [8]:
# 라벨 주요 필드 구조 확인
result = r["results"][0]
print("최상위 키:", list(result.keys())[:20])

최상위 키: ['spl_product_data_elements', 'active_ingredient', 'purpose', 'indications_and_usage', 'warnings', 'do_not_use', 'ask_doctor', 'ask_doctor_or_pharmacist', 'stop_use', 'pregnancy_or_breast_feeding', 'keep_out_of_reach_of_children', 'dosage_and_administration', 'storage_and_handling', 'inactive_ingredient', 'questions', 'package_label_principal_display_panel', 'set_id', 'id', 'effective_time', 'version']


In [9]:
# openFDA 히트율 테스트용 영어 성분명 목록 (한국 다빈도 약물)
test_names = [
    "acetaminophen", "ibuprofen", "aspirin", "metformin", "amlodipine",
    "atorvastatin", "omeprazole", "losartan", "metoprolol", "lisinopril",
    "clopidogrel", "rosuvastatin", "esomeprazole", "valsartan", "glimepiride",
    "pregabalin", "tramadol", "alprazolam", "levothyroxine", "furosemide"
]
print(f"테스트 성분명 {len(test_names)}종")

테스트 성분명 20종


In [10]:
hit_results = {}
for name in test_names[:20]:  # rate limit 고려 20종만
    r = openfda_get("/drug/label.json", {
        "search": f"openfda.generic_name:{name}",
        "limit": 1
    })
    hit = bool(r and r.get("results"))
    hit_results[name] = hit
    time.sleep(0.2)

hit_rate = sum(hit_results.values()) / len(hit_results)
print(f"히트율: {hit_rate:.1%}")
for name, hit in hit_results.items():
    print(f"  {'✓' if hit else '✗'} {name}")

히트율: 100.0%
  ✓ acetaminophen
  ✓ ibuprofen
  ✓ aspirin
  ✓ metformin
  ✓ amlodipine
  ✓ atorvastatin
  ✓ omeprazole
  ✓ losartan
  ✓ metoprolol
  ✓ lisinopril
  ✓ clopidogrel
  ✓ rosuvastatin
  ✓ esomeprazole
  ✓ valsartan
  ✓ glimepiride
  ✓ pregabalin
  ✓ tramadol
  ✓ alprazolam
  ✓ levothyroxine
  ✓ furosemide


In [11]:
# 부작용(adverse event) 건수 상위 성분
r_event = openfda_get("/drug/event.json", {
    "count": "patient.drug.openfda.generic_name.exact",
    "limit": 20
})
if r_event:
    top_ae = pd.DataFrame(r_event["results"])
    print(top_ae.head(10))

                   term   count
0            ADALIMUMAB  695827
1            ETANERCEPT  591580
2               ASPIRIN  518562
3         ASPIRIN 81 MG  493204
4        ASPIRIN 325 MG  492958
5            PREDNISONE  482870
6   METHOTREXATE SODIUM  473797
7  OMEPRAZOLE MAGNESIUM  433232
8         ACETAMINOPHEN  423831
9             DUPILUMAB  411692


In [12]:
summary = {
    "test_drug_count": len(hit_results),
    "hit_rate": float(hit_rate),
    "hit_details": hit_results
}
with open(INTERIM / "openfda_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("저장 완료")

저장 완료
